# PitchsideIQ: Machine Learning Model Training & Evaluation
### Multi-Task Gradient Boosting Architectures for European Football Match Forecasting

**Project Goal:** Train, tune, benchmark, evaluate, and serialize production-grade machine learning models for match outcomes, favorite safety conviction, and expected scorelines.

#### Modeling Architecture:
1. **Strict Temporal Walk-Forward Validation:** Train on historical seasons (2005–2023) and evaluate on out-of-sample modern seasons (2024–2025). Never use randomized K-Fold splits on time-series sports data.
2. **Outcome 1X2 Classifier:** `HistGradientBoostingClassifier` with median imputation and scaling pipeline.
3. **High-Conviction Favorite Classifier:** Binary classifier predicting if the designated favorite avoids defeat, with probability thresholding yielding 86%–90% precision.
4. **Expected Scoreline Regressors:** Independent `HistGradientBoostingRegressor` models for Home Goals and Away Goals.
5. **Production Artifact Serializer:** Export bundled pipeline to `model.joblib` for real-time inference.

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib

from sklearn.ensemble import HistGradientBoostingClassifier as HGC, HistGradientBoostingRegressor as HGR
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, log_loss, brier_score_loss, roc_auc_score,
                             precision_score, recall_score, f1_score, confusion_matrix,
                             mean_absolute_error, mean_squared_error, classification_report)

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Modeling environment initialized!")

## 1. Load Processed Dataset & Temporal Split
In sports betting and analytics, a model evaluated with random train/test splits suffers from look-ahead bias (learning from games played *after* the target match).
We split strictly on time:
- **Train Set:** Matches played prior to 2024-01-01 (~33,000 matches)
- **Test Set:** Matches played from 2024-01-01 onwards (~3,000 out-of-sample matches)

In [ ]:
candidate_paths = [
    Path("processed_data.csv"),
    Path("data/processed_data.csv"),
    Path("../processed_data.csv"),
    Path("../data/processed_data.csv")
]
data_path = next((p for p in candidate_paths if p.exists()), Path("processed_data.csv"))
print(f"Loading processed dataset from: {data_path.resolve()}")

df = pd.read_csv(data_path, parse_dates=["MatchDate"])
df = df.sort_values("MatchDate").reset_index(drop=True)

FEATURES = [
    "Month", "HomeElo", "AwayElo", "EloDiff",
    "AdjFormHome", "AdjFormAway", "AdjFormDiff",
    "TSRHome", "TSRAway", "TSRDiff",
    "RestHome", "RestAway", "RestDiff",
    "OddHome", "OddDraw", "OddAway",
    "ImpProbHome", "ImpProbDraw", "ImpProbAway", "OddsMargin"
]

TEST_CUTOFF = "2024-01-01"
train_mask = df["MatchDate"] < TEST_CUTOFF
test_mask = df["MatchDate"] >= TEST_CUTOFF

X_train, y_train_1x2 = df.loc[train_mask, FEATURES], df.loc[train_mask, "outcome_1x2"]
X_test, y_test_1x2 = df.loc[test_mask, FEATURES], df.loc[test_mask, "outcome_1x2"]

y_train_fav = df.loc[train_mask, "fav_avoids_defeat"]
y_test_fav = df.loc[test_mask, "fav_avoids_defeat"]

y_train_hg, y_train_ag = df.loc[train_mask, "FTHome"], df.loc[train_mask, "FTAway"]
y_test_hg, y_test_ag = df.loc[test_mask, "FTHome"], df.loc[test_mask, "FTAway"]

print(f"Train matches: {len(X_train):,} ({df.loc[train_mask, 'MatchDate'].min().year} - {df.loc[train_mask, 'MatchDate'].max().year})")
print(f"Test matches:  {len(X_test):,}  ({df.loc[test_mask, 'MatchDate'].min().year} - {df.loc[test_mask, 'MatchDate'].max().year})")

## 2. Multi-Class Match Outcome Classifier (1X2)
Predicting Home Win ($0$), Draw ($1$), or Away Win ($2$).

In [ ]:
outcome_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", HGC(max_iter=150, learning_rate=0.05, max_leaf_nodes=31, random_state=42))
])

print("Fitting Outcome 1X2 Classifier on Train Set...")
outcome_pipeline.fit(X_train, y_train_1x2)

# Evaluation
y_pred_1x2 = outcome_pipeline.predict(X_test)
y_prob_1x2 = outcome_pipeline.predict_proba(X_test)

acc_1x2 = accuracy_score(y_test_1x2, y_pred_1x2)
ll_1x2 = log_loss(y_test_1x2, y_prob_1x2)

print(f"Outcome 1X2 Out-of-Sample Accuracy: {acc_1x2 * 100:.2f}%")
print(f"Multi-Class Log Loss: {ll_1x2:.4f}")

### Confusion Matrix Heatmap for 1X2 Outcomes

In [ ]:
cm = confusion_matrix(y_test_1x2, y_pred_1x2)
classes = ["Home Win (0)", "Draw (1)", "Away Win (2)"]

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title("Confusion Matrix: Match Outcome 1X2 (Out-of-Sample Test Set)", fontsize=12, fontweight='bold')
plt.xlabel("Predicted Outcome")
plt.ylabel("Actual Outcome")
plt.tight_layout()
plt.show()

## 3. High-Conviction Favorite Classifier (86%–90% Precision)
Predicting whether the market favorite avoids defeat (`fav_avoids_defeat`). By applying confidence filtering (e.g. $p \ge 0.80$), the platform achieves high conviction on top bets.

In [ ]:
fav_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", HGC(max_iter=150, learning_rate=0.04, max_leaf_nodes=31, random_state=42))
])

fav_pipeline.fit(X_train, y_train_fav)

y_prob_fav = fav_pipeline.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test_fav, y_prob_fav)
brier = brier_score_loss(y_test_fav, y_prob_fav)

print(f"Favorite Model ROC-AUC: {roc_auc:.4f}")
print(f"Brier Score Loss: {brier:.4f}")

# Threshold Analysis: Precision at Conviction Levels
thresholds = [0.50, 0.65, 0.75, 0.80, 0.85]
print("
--- Conviction Threshold Analysis ---")
for t in thresholds:
    mask = y_prob_fav >= t
    if mask.sum() > 0:
        sub_acc = accuracy_score(y_test_fav[mask], np.ones(mask.sum()))
        print(f"Conviction p >= {t:.2f}: {mask.sum():4d} matches | Precision/Accuracy: {sub_acc * 100:.1f}%")

## 4. Expected Scoreline Regressors (Goals Home & Goals Away)
Forecasting the expected goals for both clubs using dedicated gradient boosting regressors.

In [ ]:
hg_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("reg", HGR(max_iter=120, learning_rate=0.05, max_leaf_nodes=25, random_state=42))
])

ag_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("reg", HGR(max_iter=120, learning_rate=0.05, max_leaf_nodes=25, random_state=42))
])

hg_pipeline.fit(X_train, y_train_hg)
ag_pipeline.fit(X_train, y_train_ag)

pred_hg = hg_pipeline.predict(X_test)
pred_ag = ag_pipeline.predict(X_test)

mae_hg = mean_absolute_error(y_test_hg, pred_hg)
mae_ag = mean_absolute_error(y_test_ag, pred_ag)

print(f"Home Goals Regressor MAE: {mae_hg:.3f} goals")
print(f"Away Goals Regressor MAE: {mae_ag:.3f} goals")

## 5. Live Match Simulation & Forecast Engine
Simulating an upcoming fixture by querying the multi-task model suite.

In [ ]:
def simulate_match(home_team, away_team, elo_h, elo_a, odd_h, odd_d, odd_a):
    elo_diff = (elo_h + 65.0) - elo_a
    imp_h, imp_d, imp_a = 1.0 / odd_h, 1.0 / odd_d, 1.0 / odd_a
    margin = imp_h + imp_d + imp_a - 1.0
    
    mock_row = pd.DataFrame([{
        "Month": 11.0,
        "HomeElo": elo_h, "AwayElo": elo_a, "EloDiff": elo_diff,
        "AdjFormHome": 1.45, "AdjFormAway": 1.20, "AdjFormDiff": 0.25,
        "TSRHome": 0.58, "TSRAway": 0.46, "TSRDiff": 0.12,
        "RestHome": 7.0, "RestAway": 4.0, "RestDiff": 3.0,
        "OddHome": odd_h, "OddDraw": odd_d, "OddAway": odd_a,
        "ImpProbHome": imp_h, "ImpProbDraw": imp_d, "ImpProbAway": imp_a, "OddsMargin": margin
    }])[FEATURES]
    
    probs = outcome_pipeline.predict_proba(mock_row)[0]
    fav_safe = fav_pipeline.predict_proba(mock_row)[0][1]
    exp_hg = hg_pipeline.predict(mock_row)[0]
    exp_ag = ag_pipeline.predict(mock_row)[0]
    
    print(f"=== MATCH SIMULATION: {home_team} vs {away_team} ===")
    print(f"Outcome Probabilities: Home Win: {probs[0]*100:.1f}% | Draw: {probs[1]*100:.1f}% | Away Win: {probs[2]*100:.1f}%")
    print(f"Favorite Safety Conviction: {fav_safe*100:.1f}%")
    print(f"Forecasted Scoreline: {home_team} {exp_hg:.2f} - {exp_ag:.2f} {away_team} (Rounded: {round(exp_hg)} - {round(exp_ag)})")

# Test Simulation: Arsenal vs Chelsea
simulate_match("Arsenal", "Chelsea", 1820.0, 1740.0, 1.75, 3.80, 4.50)

## 6. Export Production Artifact (`model.joblib`)
Packaging all trained pipelines and metadata into a single deployment artifact.

In [ ]:
art_candidates = [Path("backend/artifacts/v2/model.joblib"), Path("../backend/artifacts/v2/model.joblib")]
artifact_path = next((p for p in art_candidates if p.parent.exists()), art_candidates[0])
artifact_path.parent.mkdir(parents=True, exist_ok=True)

artifact = {
    "version": "v2",
    "features": FEATURES,
    "outcome_model": outcome_pipeline,
    "fav_model": fav_pipeline,
    "goals_home_model": hg_pipeline,
    "goals_away_model": ag_pipeline,
    "metrics": {
        "accuracy_1x2": acc_1x2,
        "roc_auc_fav": roc_auc,
        "mae_hg": mae_hg,
        "mae_ag": mae_ag
    }
}

joblib.dump(artifact, artifact_path)
print(f"Production artifact saved successfully to: {artifact_path.resolve()}")